# Lab 19-07 · Timing Budgets and Metastability

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import timing as tm

## Part 1 · Metastability

*Predict* the resolution times for offsets of 1 mV and 1 uV with tau = 100 ps, then compare with `ngspice -b meta.cir`.

In [2]:
offs = np.logspace(-9, -2, 8)
tr = [tm.resolve_time(o) for o in offs]
for o, t in zip(offs, tr):
    print(f"offset {o:.0e} V: {t * 1e12:7.1f} ps")
slope = np.polyfit(np.log(1 / offs[:5]), tr[:5], 1)[0]
print(f"fitted tau {slope * 1e12:.2f} ps; small-signal C/g_m = {tm.tau_small_signal() * 1e12:.2f} ps")
plt.semilogx(1 / offs, np.array(tr) * 1e12, "o-"); plt.xlabel("1 / offset (1/V)"); plt.ylabel("resolution time (ps)"); plt.show()

offset 1e-09 V:  2025.3 ps
offset 1e-08 V:  1795.1 ps
offset 1e-07 V:  1564.9 ps
offset 1e-06 V:  1334.6 ps
offset 1e-05 V:  1104.4 ps
offset 1e-04 V:   874.1 ps
offset 1e-03 V:   643.9 ps
offset 1e-02 V:   413.6 ps
fitted tau 99.99 ps; small-signal C/g_m = 100.00 ps


## Part 2 · What sets tau

*Predict* tau with twice the node capacitance, and with twice the transistor width.

In [3]:
for label, kw in (("baseline", {}), ("2x capacitance", {"c": 20e-15}), ("2x width", {"k": 400e-6})):
    t1, t2 = tm.resolve_time(1e-4, **kw), tm.resolve_time(1e-7, **kw)
    print(f"{label:15s}: tau = {(t2 - t1) / math.log(1000) * 1e12:.1f} ps")

baseline       : tau = 100.0 ps
2x capacitance : tau = 200.0 ps
2x width       : tau = 50.0 ps


## Part 3 · Skew windows

*Predict* the allowed skew for Example 19-07.3 (500 ps clock).

In [4]:
sk = np.linspace(-100, 100, 401)
setup = [500 - tm.min_period(30, 400, 20, skew=s) for s in sk]
hold = [tm.hold_slack(20, 30, 15, skew=s) for s in sk]
plt.plot(sk, setup, label="setup slack"); plt.plot(sk, hold, label="hold slack"); plt.axhline(0, color="k")
plt.xlabel("skew (ps)"); plt.ylabel("slack (ps)"); plt.legend(); plt.show()
ok = [s for s, a, b in zip(sk, setup, hold) if a >= 0 and b >= 0]
print(f"window: {min(ok):.1f} to {max(ok):.1f} ps")

window: -50.0 to 35.0 ps


## Part 4 · Pipelining

*Predict* how throughput and overhead share change with depth for overheads of 30, 60 and 120 ps.

In [5]:
n = np.arange(1, 33)
for to in (30, 60, 120):
    tc = 2000 / n + to
    plt.plot(n, 1e3 / tc, label=f"overhead {to} ps")
    print(f"overhead {to} ps: 1 stage {1e3 / (2000 + to):.3f} GHz, 16 stages {1e3 / (2000 / 16 + to):.2f} GHz ({100 * to / (2000 / 16 + to):.1f}% overhead)")
plt.xlabel("stages"); plt.ylabel("throughput (GHz)"); plt.legend(); plt.show()

overhead 30 ps: 1 stage 0.493 GHz, 16 stages 6.45 GHz (19.4% overhead)
overhead 60 ps: 1 stage 0.485 GHz, 16 stages 5.41 GHz (32.4% overhead)
overhead 120 ps: 1 stage 0.472 GHz, 16 stages 4.08 GHz (49.0% overhead)


## Part 5 · MTBF

*Predict* the resolution time a synchronizer needs for a 10-year MTBF (Problem 19-07.10).

In [6]:
tau, t0, fclk, fdata = 15e-12, 10e-12, 2e9, 50e6
year = 365.25 * 86400
for tr_ in (300e-12, 450e-12, 500.8e-12, 950e-12):
    print(f"t_r = {tr_ * 1e12:6.1f} ps: MTBF = {tm.mtbf(tr_, tau, t0, fclk, fdata) / year:.3g} years")

t_r =  300.0 ps: MTBF = 1.54e-05 years
t_r =  450.0 ps: MTBF = 0.339 years
t_r =  500.8 ps: MTBF = 10 years
t_r =  950.0 ps: MTBF = 1.01e+14 years


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).